In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.transforms as transforms
from sklearn.metrics import f1_score, jaccard_score
from tqdm import tqdm  # 引入进度条
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler
class UNetWithClassification(nn.Module):
    def __init__(self, in_channels, num_classes, freeze_encoder_layers=3, dropout_prob=0.5):
        super(UNetWithClassification, self).__init__()

        # 加载ResNet34作为编码器
        self.encoder = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        
        # 冻结前几层
        for param in self.encoder.parameters():
            param.requires_grad = False

        layers_to_freeze = list(self.encoder.children())[:freeze_encoder_layers]
        for layer in layers_to_freeze:
            for param in layer.parameters():
                param.requires_grad = False

        # 解码器部分
        self.decoder = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_prob),
            nn.ConvTranspose2d(128, 1, kernel_size=2, stride=2)  # 单通道输出掩码
        )

        # 分类部分
        self.fc = nn.Sequential(
            nn.Linear(512 * 7 * 7, 512),  # 输入为ResNet34最后一层的输出
            nn.ReLU(),
            nn.Linear(512, num_classes)  # 输出为类别数
        )
        
    def forward(self, x):
        # 编码器部分
        x1 = self.encoder.conv1(x)
        x1 = self.encoder.bn1(x1)
        x1 = self.encoder.relu(x1)
        x1 = self.encoder.maxpool(x1)

        x2 = self.encoder.layer1(x1)
        x3 = self.encoder.layer2(x2)
        x4 = self.encoder.layer3(x3)
        x5 = self.encoder.layer4(x4)
        
        # 解码器部分（用于分割）
        segmentation_output = self.decoder(x5)

        # 分类部分
        x5_flatten = x5.view(x5.size(0), -1)  # 展平
        classification_output = self.fc(x5_flatten)

        return segmentation_output, classification_output


class CombinedLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha  # 分类损失的权重
        self.beta = beta  # 分割损失的权重
        self.classification_loss = nn.CrossEntropyLoss()  # 分类损失
        self.segmentation_loss = nn.BCEWithLogitsLoss()  # 分割损失

    def forward(self, segmentation_output, classification_output, segmentation_target, classification_target):
        # 计算分类损失
        classification_loss = self.classification_loss(classification_output, classification_target)

        # 计算分割损失
        # 确保分割标签为float并且为0和1之间
        segmentation_target = segmentation_target.float()
        segmentation_loss = self.segmentation_loss(segmentation_output, segmentation_target)

        # 综合损失
        loss = self.alpha * classification_loss + self.beta * segmentation_loss
        return loss


class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()

    def forward(self, inputs, targets):
        smooth = 1.0
        inputs = torch.sigmoid(inputs)
        
        intersection = (inputs * targets).sum()
        dice = (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)
        return 1 - dice


def compute_f1_score(predictions, targets):
    predictions = predictions.flatten()
    targets = targets.flatten()
    return f1_score(targets, predictions, average='weighted')

def compute_miou(predictions, targets, num_classes=2):  # 这里num_classes取决于分割任务
    ious = []
    for i in range(num_classes):
        intersection = np.sum((predictions == i) & (targets == i))
        union = np.sum((predictions == i) | (targets == i))
        iou = intersection / float(union) if union != 0 else 0
        ious.append(iou)
    return np.mean(ious)

class TumorDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform_image=None, transform_mask=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform_image = transform_image
        self.transform_mask = transform_mask

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # 读取图像和标签
        image = plt.imread(self.image_paths[idx])
        mask = plt.imread(self.mask_paths[idx])

        # 如果图像是 numpy ndarray，则将其转换为 PIL Image
        if isinstance(image, np.ndarray):
            if image.dtype.kind == 'f':
                image = Image.fromarray((image * 255).astype(np.uint8))
            else:
                image = Image.fromarray(image)
        
        if isinstance(mask, np.ndarray):
            mask = np.uint8(mask * 255)  # 将标签转换为uint8
            mask = Image.fromarray(mask)  # 转换为PIL Image

        # 对图像应用转换
        if self.transform_image:
            image = self.transform_image(image)
        
        # 对标签应用转换
        if self.transform_mask:
            mask = self.transform_mask(mask)

        return image, mask


# 获取当前设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import torch.nn.functional as F

def train_model(model, train_loader, val_loader, epochs=10, learning_rate=1e-4, weight_decay=1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    criterion = CombinedLoss(alpha=1.0, beta=1.0)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    model = model.to(device)

    train_losses, train_f1, train_iou = [], [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        with tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', unit='batch') as tepoch:
            for inputs, labels in tepoch:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # 分割标签处理：去掉标签的多余通道维度
                segmentation_labels = labels.squeeze(1)  # 转换为 [batch_size, height, width]

                # 获取分类标签：从标签中提取每个图像的类别
                # 假设类别信息存储在标签的某个位置，例如根据每个样本的标签类别确定
                classification_labels = labels.view(labels.size(0), -1).max(dim=1).values.long()  # 每个图像的类别标签

                optimizer.zero_grad()

                # 获取模型输出
                segmentation_output, classification_output = model(inputs)

                # 上采样分割输出以匹配标签的大小
                segmentation_output = F.interpolate(segmentation_output, size=(224, 224), mode='bilinear', align_corners=False)

                # 调整分割标签形状
                segmentation_labels = segmentation_labels.unsqueeze(1)  # 转换为 [batch_size, 1, height, width]

                # 计算损失
                loss = criterion(segmentation_output, classification_output, segmentation_labels, classification_labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

                # 更新进度条
                tepoch.set_postfix(loss=running_loss/len(tepoch))

        train_losses.append(running_loss / len(train_loader))

        # 验证集评估
        model.eval()
        all_labels, all_preds = [], []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                segmentation_output, classification_output = model(inputs)
                segmentation_output = F.interpolate(segmentation_output, size=(224, 224), mode='bilinear', align_corners=False)

                preds = torch.argmax(classification_output, dim=1)  # 分类预测

                # 获取分类标签：每个图像对应一个标签
                classification_labels = labels.view(labels.size(0), -1).max(dim=1).values.long()  # 每个图像的类别标签

                # 拼接标签和预测
                all_labels.append(classification_labels)
                all_preds.append(preds.view(-1))  # 扁平化预测值

        # 确保分类标签和预测值的一致性
        all_labels = torch.cat(all_labels).cpu().numpy()  # 扁平化
        all_preds = torch.cat(all_preds).cpu().numpy()  # 扁平化

        # 打印所有标签和预测的形状
        print(f"All labels shape: {all_labels.shape}")
        print(f"All predictions shape: {all_preds.shape}")

        # 计算加权 F1 和 IoU
        f1 = f1_score(all_labels, all_preds, average='weighted')
        iou = jaccard_score(all_labels, all_preds, average='weighted')

        train_f1.append(f1)
        train_iou.append(iou)

        print(f'Epoch {epoch+1}/{epochs} - Loss: {train_losses[-1]:.4f}, F1 Score: {f1:.4f}, IoU: {iou:.4f}')

        scheduler.step()

    # 绘制训练曲线
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.plot(range(epochs), train_losses, label='Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss')

    plt.subplot(1, 3, 2)
    plt.plot(range(epochs), train_f1, label='F1 Score', color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.title('Training F1 Score')

    plt.subplot(1, 3, 3)
    plt.plot(range(epochs), train_iou, label='IoU', color='green')
    plt.xlabel('Epoch')
    plt.ylabel('IoU')
    plt.title('Training IoU')

    plt.tight_layout()
    plt.show()



def visualize_output(model, dataloader):
    model.eval()
    with torch.no_grad():
        for inputs, labels in dataloader:
            # 获取模型的两个输出：分割掩码和分类输出
            segmentation_output, classification_output = model(inputs)

            # 对分割输出应用 sigmoid 函数，并设置阈值
            preds = torch.sigmoid(segmentation_output) > 0.5

            plt.figure(figsize=(12, 6))

            # 显示原始图像
            plt.subplot(1, 4, 1)
            plt.imshow(inputs[0].permute(1, 2, 0))  # 显示原始图像
            plt.title('Input Image')

            # 显示标签（真实的分割掩码）
            plt.subplot(1, 4, 2)
            plt.imshow(labels[0, 0], cmap='gray')  # 只显示单通道
            plt.title('Ground Truth')

            # 显示预测的分割掩码
            plt.subplot(1, 4, 3)
            plt.imshow(preds[0].cpu().numpy(), cmap='gray')  # 显示预测结果
            plt.title('Predicted Mask')

            # 显示分类结果（良性、恶性等）
            plt.subplot(1, 4, 4)
            plt.bar([0, 1, 2], classification_output[0].cpu().detach().numpy())  # 根据分类数显示结果
            plt.title("Predicted Class")

            plt.tight_layout()
            plt.show()
            break  # 只显示一张图像

def visualize_results(inputs, segmentation_output, classification_output, ground_truth):
    plt.figure(figsize=(15, 5))

    # 原始图像
    plt.subplot(1, 4, 1)
    plt.imshow(inputs[0].permute(1, 2, 0).cpu().numpy())
    plt.title("Input Image")

    # 真实标签
    plt.subplot(1, 4, 2)
    plt.imshow(ground_truth[0, 0].cpu().numpy(), cmap='gray')  # 只显示单通道
    plt.title("Ground Truth")

    # 分割结果
    plt.subplot(1, 4, 3)
    plt.imshow(segmentation_output[0, 0].cpu().numpy(), cmap='gray')  # 只显示单通道
    plt.title("Predicted Mask")

    # 分类预测
    plt.subplot(1, 4, 4)
    plt.bar([0, 1, 2], classification_output[0].cpu().detach().numpy())  # 根据分类数显示结果
    plt.title("Predicted Class")

    plt.tight_layout()
    plt.show()

# 数据路径
image_dir_benign = "/root/autodl-tmp/Dataset_BUSI/Dataset_BUSI_with_GT/benign/"
image_dir_malignant = "/root/autodl-tmp/Dataset_BUSI/Dataset_BUSI_with_GT/malignant/"
image_dir_normal = "/root/autodl-tmp/Dataset_BUSI/Dataset_BUSI_with_GT/normal/"

mask_dir_benign = "/root/autodl-tmp/Dataset_BUSI/Dataset_BUSI_with_GT/benign_mask/"
mask_dir_malignant = "/root/autodl-tmp/Dataset_BUSI/Dataset_BUSI_with_GT/malignant_mask/"
mask_dir_normal = "/root/autodl-tmp/Dataset_BUSI/Dataset_BUSI_with_GT/normal_mask/"

# 获取所有图像路径
benign_image_paths = [os.path.join(image_dir_benign, fname) for fname in os.listdir(image_dir_benign) if fname.endswith('.png')]
malignant_image_paths = [os.path.join(image_dir_malignant, fname) for fname in os.listdir(image_dir_malignant) if fname.endswith('.png')]
normal_image_paths = [os.path.join(image_dir_normal, fname) for fname in os.listdir(image_dir_normal) if fname.endswith('.png')]

# 获取对应标签路径
benign_mask_paths = [os.path.join(mask_dir_benign, fname.replace('.png', '_mask.png')) for fname in os.listdir(image_dir_benign) if fname.endswith('.png')]
malignant_mask_paths = [os.path.join(mask_dir_malignant, fname.replace('.png', '_mask.png')) for fname in os.listdir(image_dir_malignant) if fname.endswith('.png')]
normal_mask_paths = [os.path.join(mask_dir_normal, fname.replace('.png', '_mask.png')) for fname in os.listdir(image_dir_normal) if fname.endswith('.png')]

# 合并图像和标签路径
image_paths = benign_image_paths + malignant_image_paths + normal_image_paths
mask_paths = benign_mask_paths + malignant_mask_paths + normal_mask_paths

# 使用 sklearn 进行数据集划分
train_image_paths, val_image_paths, train_mask_paths, val_mask_paths = train_test_split(image_paths, mask_paths, test_size=0.2, random_state=42)

# 数据预处理
# 数据预处理
transform_image = transforms.Compose([
    transforms.Resize((224, 224)),  # 将图像调整为固定尺寸，例如224x224
    transforms.Lambda(lambda x: x.convert("RGB") if x.mode != 'RGB' else x),  # 强制转换为RGB格式
    transforms.ToTensor(),  # 将图像转换为Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 对图像进行标准化
])

# 只对 mask 图像进行变换，不使用 transforms.Resize 直接对 tensor 进行处理
transform_mask = transforms.Compose([
    transforms.Resize((224, 224)),  # 确保标签的尺寸一致
    transforms.Grayscale(num_output_channels=1),  # 强制为单通道
    transforms.ToTensor(),
    transforms.Lambda(lambda x: torch.where(x > 0.5, torch.tensor(1.0), torch.tensor(0.0)))  # 标签二值化
])

# 创建训练集和验证集的数据集对象
train_dataset = TumorDataset(image_paths=train_image_paths, mask_paths=train_mask_paths, 
                              transform_image=transform_image, transform_mask=transform_mask)
val_dataset = TumorDataset(image_paths=val_image_paths, mask_paths=val_mask_paths, 
                            transform_image=transform_image, transform_mask=transform_mask)
# 计算样本权重
class_weights = [1/437, 1/210, 1/133]  # 这里的权重是根据每个类别的数量来设置的
sample_weights = []

# 给每个图像路径分配权重
for image_path in train_image_paths:
    if 'benign' in image_path:
        sample_weights.append(class_weights[0])
    elif 'malignant' in image_path:
        sample_weights.append(class_weights[1])
    else:
        sample_weights.append(class_weights[2])

# 使用 WeightedRandomSampler
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# 使用 sampler 创建 train_loader
train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler)

# 使用标准的顺序加载验证集
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# 初始化模型，使用 num_classes 代替 out_channels
model = UNetWithClassification(in_channels=3, num_classes=3, freeze_encoder_layers=1)  # 这里 num_classes = 3，代表有三个类别（良性、恶性、正常）

# 检查第一个样本的维度
sample_img, sample_mask = next(iter(train_loader))

print(f"Image shape: {sample_img.shape}")  # 应为 [batch, 3, 224, 224]
print(f"Mask shape: {sample_mask.shape}")  # 应为 [batch, 1, 224, 224]

# 开始训练
train_model(model, train_loader, val_loader, epochs=20, learning_rate=1e-4, weight_decay=1e-4)



Image shape: torch.Size([8, 3, 224, 224])
Mask shape: torch.Size([8, 1, 224, 224])


Epoch 1/20: 100%|██████████| 78/78 [00:13<00:00,  5.69batch/s, loss=0.87] 


All labels shape: (156,)
All predictions shape: (156,)
Epoch 1/20 - Loss: 0.8700, F1 Score: 0.8580, IoU: 0.7703


Epoch 2/20: 100%|██████████| 78/78 [00:12<00:00,  6.10batch/s, loss=0.517] 


All labels shape: (156,)
All predictions shape: (156,)
Epoch 2/20 - Loss: 0.5171, F1 Score: 0.8530, IoU: 0.7561


...

Epoch 16/20: 100%|██████████| 78/78 [00:13<00:00,  5.96batch/s, loss=0.139] 


All labels shape: (156,)
All predictions shape: (156,)
Epoch 16/20 - Loss: 0.1389, F1 Score: 0.9018, IoU: 0.8305


Epoch 17/20: 100%|██████████| 78/78 [00:13<00:00,  5.73batch/s, loss=0.14]  


All labels shape: (156,)
All predictions shape: (156,)
Epoch 17/20 - Loss: 0.1404, F1 Score: 0.8812, IoU: 0.8009


Epoch 18/20:  92%|█████████▏| 72/78 [00:12<00:01,  4.87batch/s, loss=0.129] 